In [0]:
%pip install azure-storage-blob

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.9/455.9 kB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 16.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.4.0
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.10/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-ef2630d3-b1ef-40ca-99d7-a4cc1873b581
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
import io, re, zipfile, datetime
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from azure.storage.blob import BlobServiceClient

storage_account = "vesseldatavd99"
storage_key = dbutils.secrets.get(scope="vessel-secrets", key="storage-key")

 # placeholder: paste the key only in your running notebook, never commit it
spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)

svc = BlobServiceClient(account_url=f"https://{storage_account}.blob.core.windows.net",
                        credential=storage_key)
raw = svc.get_container_client("raw")
bronze_c = svc.get_container_client("bronze")

BRONZE = f"abfss://bronze@{storage_account}.dfs.core.windows.net/ais_bronze_v2"
STAGE_PREFIX = "staging_csv/"

# Days to add. Leave as None to add every zip in raw/zip/ that is not yet in Bronze.
TARGET_DAYS = None

In [0]:
zip_names = sorted(b.name for b in raw.list_blobs(name_starts_with="zip/")
                   if b.name.endswith(".zip"))

existing = set()
if DeltaTable.isDeltaTable(spark, BRONZE):
    existing = {r[0].isoformat() for r in
                spark.read.format("delta").load(BRONZE)
                .select("source_file_date").distinct().collect()}

def tag_of(zname):
    return re.search(r"AIS_(\d{4}_\d{2}_\d{2})", zname).group(1)

todo = []
for z in zip_names:
    day = datetime.datetime.strptime(tag_of(z), "%Y_%m_%d").date().isoformat()
    if TARGET_DAYS is not None and day not in TARGET_DAYS:
        continue
    if TARGET_DAYS is None and day in existing:
        continue
    todo.append((z, day))

print("days to load:", [d for _, d in todo] or "none")

days to load: ['2019-01-17', '2019-01-18']


In [0]:
for zname, day_str in todo:
    tag = tag_of(zname)
    day = datetime.date.fromisoformat(day_str)

    data = raw.get_blob_client(zname).download_blob().readall()
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        csv_name = next(n for n in z.namelist() if n.lower().endswith(".csv"))
        csv_bytes = z.read(csv_name)
    del data

    stage_name = f"{STAGE_PREFIX}AIS_{tag}.csv"
    bronze_c.upload_blob(name=stage_name, data=csv_bytes, overwrite=True, timeout=3600)
    del csv_bytes

    df = (spark.read.option("header", True).option("inferSchema", False)
          .csv(f"abfss://bronze@{storage_account}.dfs.core.windows.net/{stage_name}")
          .withColumn("source_file", F.col("_metadata.file_name"))
          .withColumn("source_file_date", F.lit(day))
          .withColumn("ingestion_id", F.lit(f"batch_{tag}"))
          .withColumn("ingestion_timestamp", F.current_timestamp()))

    # Replace only this day's partition; other dates are untouched
    writer = df.write.format("delta").partitionBy("source_file_date").mode("overwrite")
    if DeltaTable.isDeltaTable(spark, BRONZE):
        writer = writer.option("replaceWhere", f"source_file_date = '{day_str}'")
    writer.save(BRONZE)

    bronze_c.delete_blob(stage_name)
    print("bronze written:", day_str, flush=True)

bronze written: 2019-01-17
bronze written: 2019-01-18


In [0]:
bronze = spark.read.format("delta").load(BRONZE)
display(bronze.groupBy("source_file_date").count().orderBy("source_file_date"))
print(f"Total: {bronze.count():,}")

source_file_date,count
2019-01-01,7516408
2019-01-02,7169185
2019-01-03,6724018
2019-01-04,6951276
2019-01-05,6851182
2019-01-06,6968635
2019-01-07,7027781
2019-01-08,7129984
2019-01-09,7279296
2019-01-10,7104760


Total: 123,809,259
